Things to add/areas for improvement
- simulation time step -> 1hr?, 30 mins? 
-  realistic load/generation profiles for houses
      - solar curves
      - battery charge/discharge profliles
      - EV charge/discharge profiles
      - these might be found from realworld sources/data tables, find smart meter data if possible
      - should consider days where temperature and weather changes increase demand or decrease generation
- realistic load profiles for industry/hospital
      - try and find realworld data to ensure daily curves are accurate
      - need more research to ensure realism
- Improve Windturbine simulation
      - Different weather profiles that show no wind/lockout, moderate and ideal wind (can use to show DFIG behaviour)
- can this each simulation run based on changning input settings?
- data outputs that can be represented graphically and physically with leds etc
     - night/day time shift and change -> can this be visibly shown on display? e.g street lamps etc
     - 
- 

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import os


#create results folder if it doesn't exist
os.makedirs("results", exist_ok=True)


print("Environment working")


Environment working


Define simulation Objects


In [42]:
class House:
    def __init__(self, name, load_profile, generation_profile):
        self.name = name
        self.load_profile = load_profile
        self.generation_profile = generation_profile

    def step(self, t):
        load_kw = self.load_profile[t]
        generation_kw = self.generation_profile[t]
        net_power_kw = load_kw - generation_kw

        return {
            "name": self.name,
            "load_kw": load_kw,
            "generation_kw": generation_kw,
            "net_power_kw": net_power_kw,
        }
#how do I improve load and generation simulation? need realistic load and genereation profiles for each house depending on their availiable resources. could also reseach files to obtain accurrate hourly load profiles (clean data in Excel)

# Create the Battery Class/Object, This may later be moved inside of the house class once tested for behaviour

class Battery:
    def __init__(self, capacity_kwh, initial_soc_kwh, max_charge_kw, max_discharge_kw, efficiency=0.95, min_soc_kwh=0.0):
        self.capacity_kwh = capacity_kwh
        self.soc_kwh = initial_soc_kwh
        self.max_charge_kw = max_charge_kw
        self.max_discharge_kw = max_discharge_kw
        self.efficiency = efficiency
        self.min_soc_kwh = min_soc_kwh
        
    def charge(self, requested_power_kw, timestep_hours):
        # check requested charge power is positive
        if requested_power_kw <= 0:
            return 0.0
        # limit requested power
        charge_power_kw = min(requested_power_kw, self.max_charge_kw)
        # convert to stored energy
        requested_energy_kwh = charge_power_kw *timestep_hours*self.efficiency
        # check availiable battery space
        available_space_kwh = self.capacity_kwh - self.soc_kwh
        # limit actual stored energy to within max capacity
        actual_energy_kwh = min(requested_energy_kwh, available_space_kwh)
        # update the SOC
        self.soc_kwh += actual_energy_kwh
        # calculate the charging power
        actual_charge_power_kw = actual_energy_kwh / (timestep_hours*self.efficiency)
        #return value of power stored by the battery to the simulation
        return actual_charge_power_kw
        
    def discharge(self, requested_power_kw, timestep_hours):
        pass   
    def get_soc_percent(self):
        return self.soc_kwh


class LocalLoad:
    def __init__(self, name, demand_kw):
        self.name = name
        self.demand_kw = demand_kw

    def step(self, t):
        return {
            "name": self.name,
            "load_kw": self.demand_kw
        }


class WindTurbine:
    def __init__(self, name, rated_kw):
        self.name = name
        self.rated_kw = rated_kw

    def step(self, t):
        generation_kw = 0.0  # placeholder for now
        return {
            "name": self.name,
            "generation_kw": generation_kw
        }

#wind turbine could have three different generation settings based on the wind speed (no wind/lock out (too much wind), moderate wind, ideal wind)
#middle setting could have variance to show slight hour to hour changes perhaps?
#need to confirm how this works in realworld -> further research required



In [ ]:
Simulation Loop

In [17]:
# PLACEHOLDER LOAD/GENERATION PROFILES UNTIL REAL VALUES ARE CLEANED AND SELECTED

# these load profiles will be replaced with real world data from the smart grid smart city dataset

house_1_load = [0.7, 0.6, 0.6, 0.5, 0.6, 0.8, 1.4, 1.8, 1.2, 1.0, 0.9, 0.8,
                0.9, 1.0, 1.1, 1.3, 1.8, 2.6, 2.9, 2.4, 1.8, 1.3, 1.0, 0.8]

house_2_load = [0.9, 0.8, 0.7, 0.7, 0.8, 1.0, 1.8, 2.2, 1.5, 1.2, 1.1, 1.0,
                1.0, 1.1, 1.3, 1.6, 2.2, 3.1, 3.4, 2.8, 2.1, 1.6, 1.2, 1.0]

house_3_load = [0.5, 0.5, 0.4, 0.4, 0.5, 0.6, 1.0, 1.3, 0.9, 0.8, 0.7, 0.7,
                0.8, 0.8, 0.9, 1.0, 1.4, 2.0, 2.2, 1.8, 1.3, 1.0, 0.8, 0.6]

house_4_load = [1.0, 0.9, 0.8, 0.8, 0.9, 1.2, 2.0, 2.5, 1.8, 1.5, 1.3, 1.2,
                1.3, 1.4, 1.6, 1.9, 2.6, 3.7, 4.0, 3.3, 2.5, 1.9, 1.4, 1.1]

house_5_load = [0.8, 0.7, 0.7, 0.6, 0.7, 0.9, 1.6, 2.0, 1.4, 1.1, 1.0, 0.9,
                1.0, 1.1, 1.2, 1.5, 2.0, 2.9, 3.1, 2.6, 2.0, 1.5, 1.1, 0.9]
# these generation profiles will be replaced with PV generation profiles generated using an external tool such as SAM to represent different size systems and weather conditions

pv_sunny_large = [0, 0, 0, 0, 0, 0.2, 0.8, 1.8, 3.0, 4.0, 4.8, 5.2,
             5.0, 4.5, 3.6, 2.4, 1.2, 0.3, 0, 0, 0, 0, 0, 0]

pv_sunny_med = [x*0.5 for x in pv_sunny_large]

pv_cloudy = [0, 0, 0, 0, 0, 0.1, 0.4, 0.9, 1.5, 2.0, 2.3, 2.5,
             2.2, 2.0, 1.6, 1.0, 0.5, 0.1, 0, 0, 0, 0, 0, 0]

pv_none = [0]*24

#these values will change later as the smart grid smart city dataset is in half hourly readings but overall goal is for each profile to simulate a full day to show arbitrage effect of peak hour demand

profile_duration_hours = 24 #this will probably remain unchanged 

# following code ensures calculations later in simulations remain correct if length of datasets change but each profile should alwuts represent 1 day
num_steps = len(house_1_load)
timestep_hours = profile_duration_hours / num_steps
simulation_hours = num_steps * timestep_hours

#need to add Wind Turbine and industrial load profiles here potentially? UPDATE WIND GEN AND LOCAL LOAD CLASS WHEN THIS IS DONE




# Verify all profiles are the same length
profiles = [
    house_1_load,
    house_2_load,
    house_3_load,
    house_4_load,
    house_5_load,
    pv_sunny_large,
    pv_sunny_med,
    pv_cloudy,
    pv_none]

profile_lengths = set([len(profile) for profile in profiles])


if len(profile_lengths) > 1 :
    raise ValueError("Profiles are not the same length")
else :
    print("All Load and Generation Profiles are the same length")

print("Number of Simulation steps=", num_steps)
print("Number of Hours Simulated=", simulation_hours) 

All Load and Generation Profiles are the same length
Number of Simulation steps= 24
Number of Hours Simulated= 24.0


In [28]:
# create empty list to store results
simulation_results = []

#create empty counter
count = 0

# Initialise houses
houses = [
    House("House 1", house_1_load, pv_sunny_large),
    House("House 2", house_2_load, pv_sunny_med),
    House("House 3", house_3_load, pv_sunny_large),
    House("House 4", house_4_load, pv_sunny_med),
    House("House 5", house_5_load, pv_none),
]
# Initialise hospital and wind turbine

hospital = LocalLoad("Hospital", 8.0)
wind = WindTurbine("DFIG Wind Placeholder", 15.0)

for t in range(num_steps):
    count = count + 1
    house_results = [house.step(t) for house in houses]
    hospital_result = hospital.step(t)
    wind_result = wind.step(t)

    total_house_demand_kw = sum(h["load_kw"] for h in house_results)
    total_house_generation_kw = sum(h["generation_kw"] for h in house_results)
    total_house_net_power_kw = sum(h["net_power_kw"] for h in house_results)
    
    total_local_load = hospital_result["load_kw"]
    total_wind_gen = wind_result["generation_kw"]

    grid_power_kw = total_house_net_power_kw + total_local_load - total_wind_gen

    timestep_result = {
                        "Timestep": t,
                        "House_net_power_kw":  total_house_net_power_kw,
                        "House_demand_kw": total_house_demand_kw,
                        "House_Generation_kw": total_house_generation_kw,
                        "Hospital_load_kw": total_local_load,
                        "Wind_Generation_kw": total_wind_gen,
                        "Grid_power_kw": grid_power_kw
                      }
                       
    simulation_results.append(timestep_result)

# convert the simulation results to a csv and export them
simres = pd.DataFrame(data=simulation_results)

# create timestape filename
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
filename = f"SimulationResults_{timestamp}.csv" 

simres = simres.round(2)
simres.to_csv(f"results/{filename}", index=False)

# verify simulation ran correct amount of expected steps and results were exported
print(count,"simulation steps completed")  
print(f"Results exported to results/{filename}")


24 simulation steps completed
Results exported to results/SimulationResults_2026-05-20_21-12.csv


In [46]:
# battery class testing
battery_test = Battery(
    capacity_kwh=10,
    initial_soc_kwh=5.0,
    max_charge_kw=2.0,
    max_discharge_kw=2.0,
    efficiency=0.95,
    min_soc_kwh=1.0
)


timestep_hours = 1.0

battery_test.charge(2.0, timestep_hours)
print(battery_test.get_soc_percent())

# battery.discharge(4.0, timestep_hours)
# print(battery.get_soc_percent())

# print(battery.get_soc_percent())



6.9
